# 08 — Cost-Sensitive Threshold Decision

**P03: Hotel-booking cancellations and the overbooking decision**

This notebook completes the decision stage required by the project brief. It uses a leakage-free Random Forest, obtains cancellation probabilities on the held-out test set, states a relative business cost for the two error types, sweeps probability thresholds, and recommends **one threshold** that minimises the stated expected cost.

> Decision scope: the required output is a **probability threshold**, not a room-capacity simulator. The cost values below are relative scenario units because the dataset does not contain the hotel's actual financial cost of an empty room or of walking a guest.


## 1. Error Meanings and Cost Model

The final classifier produces a cancellation probability, but the probability threshold used to trigger an operational action is a separate decision.

For this project:

- **False positive (FP):** predict cancellation when the booking would actually be honoured.
- **False negative (FN):** predict no cancellation when the booking actually cancels.

We evaluate an explicit relative-cost scenario:

- FP cost = 2 relative units
- FN cost = 1 relative unit

These values are **decision assumptions**, not measured hotel financial costs. The recommended threshold is therefore conditional on this 2:1 cost ratio and should be recalculated if real operational costs differ.

The probability threshold is also **not the number of rooms to overbook**. It determines which bookings are classified as sufficiently cancellation-prone under the stated decision rule. Translating probabilities into an inventory-level overbooking quantity would require additional capacity and economic information.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.project_utils import (
    RANDOM_STATE,
    CV_FOLDS,
    TARGET,
    BOOKING_TIME_EXCLUSIONS,
    prepare_model_data,
    stratified_train_test_split,
    make_cv,
    make_random_forest,
    build_preprocessors,
)

# Relative decision-cost scenario.
# These are modelling assumptions, not measured hotel financial costs.
FP_COST = 2
FN_COST = 1

CV = make_cv()

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Random state:", RANDOM_STATE)
print("CV folds:", CV_FOLDS)
print(
    f"Relative cost assumption: "
    f"FP={FP_COST}, FN={FN_COST}"
)


## 2. Use the Canonical Booking-Time Dataset

Notebook 08 consumes the cleaned dataset generated by Notebook 03 and applies the same shared booking-time feature transformation used in Notebooks 05–07.

No independent raw-data cleaning is repeated here.


In [ ]:
DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "hotel_bookings_cleaned.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Processed dataset not found. Run Notebook 03 first."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Cleaned shape:", df.shape)

assert df.shape == (119209, 30)


In [ ]:
data = prepare_model_data(df)

forbidden = [
    "reservation_status",
    "reservation_status_date",
    *BOOKING_TIME_EXCLUSIONS,
]

remaining_forbidden = [
    c for c in forbidden
    if c in data.columns
]

print("Model-ready shape:", data.shape)
print(
    "Forbidden modelling columns remaining:",
    remaining_forbidden,
)
print(
    "Missing values:",
    int(data.isna().sum().sum()),
)

assert data.shape == (119209, 26)
assert not remaining_forbidden
assert int(data.isna().sum().sum()) == 0


## 3. Reproduce the Canonical Held-Out Split

The same deterministic 80/20 stratified split is reconstructed with `random_state=42`.

The held-out test partition is kept completely outside threshold selection. Threshold selection uses only training-set out-of-fold predictions.


In [ ]:
X = data.drop(columns=[TARGET])
y = data[TARGET].astype(int)

X_train, X_test, y_train, y_test = (
    stratified_train_test_split(X, y)
)

print("Train:", X_train.shape)
print("Held-out test:", X_test.shape)
print(
    "Cancellation rate — train:",
    round(y_train.mean(), 4),
)
print(
    "Cancellation rate — test :",
    round(y_test.mean(), 4),
)
print(
    "Held-out test is not used for threshold selection."
)

assert X_train.shape == (95367, 25)
assert X_test.shape == (23842, 25)


## 4. Canonical Final Decision Model

Random Forest is carried forward because it ranked first in both training cross-validation and held-out model evaluation.

Notebook 08 imports the same preprocessing policy and canonical 200-tree Random Forest definition used in Notebooks 06 and 07.


In [ ]:
(
    preprocessor,
    _boost_preprocessor,
    num_cols,
    cat_cols,
) = build_preprocessors(X_train)

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", make_random_forest()),
])

rf = final_model.named_steps["model"]

print(
    "Country policy: categories below 1% "
    "training frequency -> Other"
)
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print(
    "Final decision model: Random Forest,",
    rf.n_estimators,
    "trees",
)

assert len(num_cols) == 16
assert len(cat_cols) == 9
assert rf.n_estimators == 200


## 5. Define Threshold-Level Decision Metrics

For each candidate probability threshold we compute the confusion counts, standard classification metrics, and the explicit relative decision cost:

`Expected Cost = FP_COST × FP + FN_COST × FN`

The conventional 0.50 threshold is retained as the baseline for comparison.


In [ ]:
def threshold_metrics(
    y_true,
    probabilities,
    threshold,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
):
    pred = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1],
    ).ravel()

    return {
        "Threshold": float(threshold),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Accuracy":
            accuracy_score(y_true, pred),
        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0,
            ),
        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0,
            ),
        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0,
            ),
        "Expected_Cost":
            fp_cost * fp + fn_cost * fn,
    }


In [ ]:
print(
    "Generating 5-fold out-of-fold "
    "probabilities on training data..."
)

oof_prob = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=CV,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

OOF_ROC_AUC = roc_auc_score(
    y_train,
    oof_prob,
)

OOF_AP = average_precision_score(
    y_train,
    oof_prob,
)

print(
    "OOF ROC-AUC:",
    round(OOF_ROC_AUC, 4),
)
print(
    "OOF Average Precision:",
    round(OOF_AP, 4),
)
print(
    "Threshold selection uses training "
    "OOF predictions only."
)

assert len(oof_prob) == len(y_train)


## 6. Training-Only OOF Threshold Selection

Five-fold stratified out-of-fold probabilities are generated for the training partition.

Thresholds from 0.01 through 0.99 are evaluated using those OOF predictions only. The threshold with minimum expected cost is selected before the held-out test probabilities are examined.

If multiple thresholds have the same minimum cost, the deterministic tie-break chooses the threshold closest to 0.50 and then the lower threshold if necessary.


In [ ]:
thresholds = np.round(
    np.arange(0.01, 1.00, 0.01),
    2,
)

rows = [
    threshold_metrics(
        y_train,
        oof_prob,
        t,
        fp_cost=FP_COST,
        fn_cost=FN_COST,
    )
    for t in thresholds
]

threshold_results = pd.DataFrame(rows)

assert len(threshold_results) == 99
assert threshold_results.isna().sum().sum() == 0

min_cost = threshold_results[
    "Expected_Cost"
].min()

best_rows = threshold_results.loc[
    threshold_results["Expected_Cost"] == min_cost
].copy()

# Deterministic tie-break:
# among equal-cost thresholds choose closest to 0.50,
# then choose the lower threshold if still tied.
best_rows["distance_from_050"] = (
    best_rows["Threshold"] - 0.50
).abs()

best = best_rows.sort_values(
    [
        "distance_from_050",
        "Threshold",
    ]
).iloc[0]

BEST_THRESHOLD = float(
    best["Threshold"]
)

print("\nTraining-only threshold selection:")
print(
    "Recommended threshold:",
    BEST_THRESHOLD,
)
print(
    "Minimum OOF expected cost:",
    int(best["Expected_Cost"]),
)

print("\nTen lowest-cost OOF thresholds:")
print(
    threshold_results
    .sort_values(
        ["Expected_Cost", "Threshold"]
    )
    .head(10)
    .round(4)
    .to_string(index=False)
)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    threshold_results["Threshold"],
    threshold_results["Expected_Cost"],
)

ax.axvline(
    0.50,
    linestyle="--",
    label="Default threshold = 0.50",
)

ax.axvline(
    BEST_THRESHOLD,
    linestyle="--",
    label=(
        "Recommended threshold = "
        f"{BEST_THRESHOLD:.2f}"
    ),
)

ax.scatter(
    [BEST_THRESHOLD],
    [best["Expected_Cost"]],
    zorder=3,
)

ax.set_xlabel(
    "Cancellation probability threshold"
)
ax.set_ylabel(
    "OOF expected cost (relative units)"
)
ax.set_title(
    "Training OOF Cost Sweep — "
    f"FP cost={FP_COST}, FN cost={FN_COST}"
)
ax.legend()

fig.tight_layout()

cost_curve_path = (
    FIGURES_DIR
    / "threshold_cost_curve.png"
)

fig.savefig(
    cost_curve_path,
    dpi=180,
    bbox_inches="tight",
)

print("Saved:", cost_curve_path)
plt.show()


## 7. One Held-Out Decision Evaluation

After the threshold is fixed from training OOF predictions, the canonical Random Forest is fitted on all training observations.

The held-out test set is then used once to compare the conventional 0.50 decision with the pre-selected recommended threshold.


In [ ]:
print(
    "Fitting final Random Forest "
    "on all training data..."
)

final_model.fit(X_train, y_train)

y_prob = final_model.predict_proba(
    X_test
)[:, 1]

HELDOUT_ROC_AUC = roc_auc_score(
    y_test,
    y_prob,
)

HELDOUT_AP = average_precision_score(
    y_test,
    y_prob,
)

print(
    "Held-out ROC-AUC:",
    round(HELDOUT_ROC_AUC, 4),
)
print(
    "Held-out Average Precision:",
    round(HELDOUT_AP, 4),
)

baseline = threshold_metrics(
    y_test,
    y_prob,
    0.50,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
)

recommended = threshold_metrics(
    y_test,
    y_prob,
    BEST_THRESHOLD,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
)

comparison = pd.DataFrame(
    [baseline, recommended],
    index=[
        "Default 0.50",
        "Recommended",
    ],
)

comparison_cols = [
    "Threshold",
    "TN",
    "FP",
    "FN",
    "TP",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Expected_Cost",
]

print("\nHeld-out decision comparison:")
print(
    comparison[comparison_cols]
    .round(4)
    .to_string()
)

cost_reduction = (
    baseline["Expected_Cost"]
    - recommended["Expected_Cost"]
)

cost_reduction_pct = (
    100
    * cost_reduction
    / baseline["Expected_Cost"]
    if baseline["Expected_Cost"] > 0
    else 0
)

print(
    f"\nFINAL RECOMMENDED THRESHOLD: "
    f"{BEST_THRESHOLD:.2f}"
)
print(
    "Expected test cost at 0.50:",
    baseline["Expected_Cost"],
)
print(
    "Expected test cost at recommendation:",
    recommended["Expected_Cost"],
)
print(
    "Test cost reduction vs 0.50:",
    f"{cost_reduction} relative units "
    f"({cost_reduction_pct:.2f}%)",
)

assert (
    baseline["TN"]
    + baseline["FP"]
    + baseline["FN"]
    + baseline["TP"]
) == len(y_test)

assert (
    recommended["TN"]
    + recommended["FP"]
    + recommended["FN"]
    + recommended["TP"]
) == len(y_test)


In [ ]:
recommended_pred = (
    y_prob >= BEST_THRESHOLD
).astype(int)

cm_recommended = confusion_matrix(
    y_test,
    recommended_pred,
    labels=[0, 1],
)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    cm_recommended,
    display_labels=[
        "Not Cancelled",
        "Cancelled",
    ],
).plot(
    values_format="d",
    ax=ax,
)

ax.set_title(
    "Random Forest — Held-Out Confusion Matrix "
    f"at Recommended Threshold "
    f"{BEST_THRESHOLD:.2f}"
)

fig.tight_layout()

recommended_cm_path = (
    FIGURES_DIR
    / "recommended_threshold_confusion.png"
)

fig.savefig(
    recommended_cm_path,
    dpi=180,
    bbox_inches="tight",
)

print("Saved:", recommended_cm_path)
plt.show()

print(
    "Final held-out confusion matrix:"
)
print(cm_recommended)


## 8. Precision–Recall Trade-Off

Changing the threshold changes the operational balance between false alarms and missed cancellations. The OOF precision/recall curves make that trade-off visible around the selected decision point.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    label="OOF Precision",
)

ax.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    label="OOF Recall",
)

ax.axvline(
    BEST_THRESHOLD,
    linestyle="--",
    label=(
        f"Recommended = "
        f"{BEST_THRESHOLD:.2f}"
    ),
)

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.02)
ax.set_title(
    "Training OOF Precision and Recall "
    "Across Thresholds"
)
ax.legend()

fig.tight_layout()

tradeoff_path = (
    FIGURES_DIR
    / "threshold_precision_recall_tradeoff.png"
)

fig.savefig(
    tradeoff_path,
    dpi=180,
    bbox_inches="tight",
)

print("Saved:", tradeoff_path)
plt.show()


## 9. Interpretation and Limitations

The recommended threshold is conditional on the stated relative FP/FN costs. It is not a universal hotel-industry threshold and should be recalculated when operational economics change.

The model identifies predictive associations rather than causal effects. The dataset also represents historical booking behaviour from a specific setting and period, so performance may change under different hotels, markets, policies, or booking channels.

A complete production overbooking system would additionally require inventory capacity, room-level constraints, realised revenue/loss estimates, probability calibration monitoring, and operational safeguards.


## 10. Persist Decision Evidence

The complete training-OOF threshold sweep, final decision summary, cost curve, threshold precision/recall trade-off, and recommended-threshold confusion matrix are saved under `results/` for reproducibility.


In [ ]:
threshold_path = (
    RESULTS_DIR
    / "threshold_sweep.csv"
)

threshold_results.to_csv(
    threshold_path,
    index=False,
)

decision_summary = pd.DataFrame([
    {
        "Model": "Random Forest",
        "FP_Cost": FP_COST,
        "FN_Cost": FN_COST,
        "OOF_ROC_AUC": OOF_ROC_AUC,
        "OOF_Average_Precision": OOF_AP,
        "Recommended_Threshold":
            BEST_THRESHOLD,
        "Minimum_OOF_Expected_Cost":
            int(best["Expected_Cost"]),
        "Heldout_ROC_AUC":
            HELDOUT_ROC_AUC,
        "Heldout_Average_Precision":
            HELDOUT_AP,
        "Default_Threshold": 0.50,
        "Default_Test_Cost":
            baseline["Expected_Cost"],
        "Recommended_Test_Cost":
            recommended["Expected_Cost"],
        "Test_Cost_Reduction":
            cost_reduction,
        "Test_Cost_Reduction_Pct":
            cost_reduction_pct,
        "Recommended_TN":
            recommended["TN"],
        "Recommended_FP":
            recommended["FP"],
        "Recommended_FN":
            recommended["FN"],
        "Recommended_TP":
            recommended["TP"],
    }
])

decision_path = (
    RESULTS_DIR
    / "decision_summary.csv"
)

decision_summary.to_csv(
    decision_path,
    index=False,
)

print(
    "Saved training OOF threshold sweep:",
    threshold_path,
)
print(
    "Saved final decision summary:",
    decision_path,
)

required_artifacts = [
    threshold_path,
    decision_path,
    FIGURES_DIR / "threshold_cost_curve.png",
    FIGURES_DIR / "recommended_threshold_confusion.png",
    FIGURES_DIR / "threshold_precision_recall_tradeoff.png",
]

assert all(
    path.exists()
    for path in required_artifacts
)

print(
    "PASS: threshold-decision artifacts "
    "were written."
)
